<a href="https://colab.research.google.com/github/GuangxingHan/tips/blob/add-segmentation-notebook/TIPS_foreground_segmentation_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright 2026 Google LLC.

SPDX-License-Identifier: Apache-2.0

In [ ]:
# @title TIPS foreground_segmentation Demo notebook

#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at

#  https://www.apache.org/licenses/LICENSE-2.0

#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.

# Training a Foreground Segmentation Tool with TIPS

In this tutorial, we will train a linear foreground segmentation model using TIPS features.

In [ ]:
import io
import os
import pickle
import tarfile
import urllib

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
import torch
import torchvision.transforms.functional as TF
from tqdm import tqdm

In [ ]:
# @title Install dependencies and clone TIPS repo
import os
import sys

# Root directory for all files (Colab default is /content).
ROOT_DIR = os.getcwd()
TIPS_DIR = os.path.join(ROOT_DIR, 'tips')

# Install required packages.
!pip install -q torch torchvision torchaudio
!pip install -q tensorflow_text mediapy jax jaxlib scikit-learn

# Clone the TIPS repository.
if not os.path.exists(TIPS_DIR):
  !git clone https://github.com/google-deepmind/tips.git {TIPS_DIR}

# Add the root directory to PYTHONPATH so that `tips.*` imports work.
if ROOT_DIR not in sys.path:
  sys.path.insert(0, ROOT_DIR)

print(f'ROOT_DIR: {ROOT_DIR}')
print(f'TIPS_DIR: {TIPS_DIR}')
print('Installation complete!')

In [ ]:
import ipywidgets as widgets
from IPython.display import display

#@title Select Model and Variant to Use
variants_map = {
    'TIPS': ['S', 'B', 'L', 'So400m', 'g'],
    'TIPS_V2': ['B', 'L', 'So', 'g']
}
# Models Selection
model_dropdown = widgets.ToggleButtons(
    options=list(variants_map.keys()),
    description='Model Type: ',
)

variant_dropdown = widgets.Dropdown(
    options=list(variants_map['TIPS']),
    value='S',
    description='Variant: ',
)

def on_model_change(change):
    # Update the options of the variant dropdown
    new_model = change['new']
    variant_dropdown.options = variants_map[new_model]
    # Set a default value from the new options
    variant_dropdown.value = variants_map[new_model][0]

def on_variant_change(change):
    # Update the options of the variant dropdown
    new_variant = change['new']
    # Set a default value from the new options
    variant_dropdown.value = new_variant

model_dropdown.observe(on_model_change, names='value')
variant_dropdown.observe(on_variant_change, names='value')

display(model_dropdown)
display(variant_dropdown)

In [ ]:
# @title Download checkpoints and sample images
import urllib.request

CHECKPOINT_BASE_URL = {'TIPS': 'https://storage.googleapis.com/tips_data/v1_0/checkpoints/pytorch',
                       'TIPS_V2': 'https://storage.googleapis.com/tips_data/v2_0/checkpoints/pytorch'}
TOKENIZER_URL = 'https://storage.googleapis.com/tips_data/v1_0/checkpoints/tokenizer.model'
IMAGE_BASE_URL = 'https://raw.githubusercontent.com/google-deepmind/tips/main/scenic/images'

# Checkpoint files for each variant.
V1_CHECKPOINT_MAP = {
    'S': ('tips_oss_s14_highres_distilled_vision.npz', 'tips_oss_s14_highres_distilled_text.npz'),
    'B': ('tips_oss_b14_highres_distilled_vision.npz', 'tips_oss_b14_highres_distilled_text.npz'),
    'L': ('tips_oss_l14_highres_distilled_vision.npz', 'tips_oss_l14_highres_distilled_text.npz'),
    'So400m': ('tips_oss_so400m14_highres_largetext_distilled_vision.npz', 'tips_oss_so400m14_highres_largetext_distilled_text.npz'),
    'g': ('tips_oss_g14_highres_vision.npz', 'tips_oss_g14_highres_text.npz'),
}
V2_CHECKPOINT_MAP = {
    'B': ('tips_v2_oss_b14_vision.npz', 'tips_v2_oss_b14_text.npz'),
    'L': ('tips_v2_oss_l14_vision.npz', 'tips_v2_oss_l14_text.npz'),
    'So': ('tips_v2_oss_so14_vision.npz', 'tips_v2_oss_so14_text.npz'),
    'g': ('tips_v2_oss_g14_vision.npz', 'tips_v2_oss_g14_text.npz'),
}
CHECKPOINT_MAP = {
    'TIPS': V1_CHECKPOINT_MAP,
    'TIPS_V2': V2_CHECKPOINT_MAP,
}

# Model constructor map.
V1_MODEL_CONSTRUCTOR_MAP = {
    'S': 'vit_small',
    'B': 'vit_base',
    'L': 'vit_large',
    'So400m': 'vit_so400m',
    'g': 'vit_giant2',
}
V2_MODEL_CONSTRUCTOR_MAP = {
    'B': 'vit_base',
    'L': 'vit_large',
    'So': 'vit_so',
    'g': 'vit_giant2',
}
MODEL_CONSTRUCTOR_MAP = {
    'TIPS': V1_MODEL_CONSTRUCTOR_MAP,
    'TIPS_V2': V2_MODEL_CONSTRUCTOR_MAP,
}

V1_MODEL_TO_NUM_LAYERS = {
    'S': 12,
    'B': 12,
    'L': 24,
    'So400m': 27,
    'g': 40,
}
V2_MODEL_TO_NUM_LAYERS = {
    'B': 12,
    'L': 24,
    'So': 27,
    'g': 40,
}
MODEL_LAYERS_MAP = {
    'TIPS': V1_MODEL_TO_NUM_LAYERS,
    'TIPS_V2': V2_MODEL_TO_NUM_LAYERS,
}

# Select variant.
model = model_dropdown.value
variant = variant_dropdown.value

# Directories for checkpoints and images (under ROOT_DIR).
CKPT_DIR = os.path.join(ROOT_DIR, 'checkpoints')
IMG_DIR = os.path.join(ROOT_DIR, 'images')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

# Download checkpoints for selected variant.
vision_ckpt_name, text_ckpt_name = CHECKPOINT_MAP[model][variant]

for ckpt_name in [vision_ckpt_name, text_ckpt_name]:
  ckpt_path = os.path.join(CKPT_DIR, ckpt_name)
  if not os.path.exists(ckpt_path):
    print(f'\nDownloading {ckpt_name}...')
    urllib.request.urlretrieve(f'{CHECKPOINT_BASE_URL[model]}/{ckpt_name}', ckpt_path)
    print(f'  Saved to {ckpt_path}')
  else:
    print(f'  {ckpt_name} already exists.')

# Download tokenizer.
tokenizer_file = os.path.join(CKPT_DIR, 'tokenizer.model')
if not os.path.exists(tokenizer_file):
  print('\nDownloading tokenizer...')
  urllib.request.urlretrieve(TOKENIZER_URL, tokenizer_file)
  print(f'  Saved to {tokenizer_file}')
else:
  print('  tokenizer.model already exists.')

# Download sample images.
sample_images = ['example_image.jpg', 'example_image_2.jpg']
for img_name in sample_images:
  img_path = os.path.join(IMG_DIR, img_name)
  if not os.path.exists(img_path):
    print(f'\nDownloading {img_name}...')
    urllib.request.urlretrieve(f'{IMAGE_BASE_URL}/{img_name}', img_path)
    print(f'  Saved to {img_path}')
  else:
    print(f'  {img_name} already exists.')

print('\nAll downloads complete!')

### Imports and functions

Please follow the installation guide before you run the following cells.

In [ ]:
import glob
import io
import mediapy as media
import numpy as np
from PIL import Image
import tensorflow_text
from tips.pytorch import image_encoder
from tips.pytorch import text_encoder
from tips.scenic.utils import feature_viz
import torch
import torch.nn.functional as F
from torchvision import transforms

IMAGE_MEAN = (0, 0, 0)
IMAGE_STD = (1.0, 1.0, 1.0)
PATCH_SIZE = 14
MAX_LEN = 64
VOCAB_SIZE = 32000


### Configure the TIPS model.

In [ ]:
# Set the input image shape and variant.
image_size = 448  # @param {type: "number"}
# variant is already set in the download cell above.

# Image directory (absolute path).
image_dir = IMG_DIR
image_paths = sorted(glob.glob(os.path.join(image_dir, '*')))
print(f'Found {len(image_paths)} images: {image_paths}')

# The text inputs to be contrasted.
text_inputs = [
    'A ship',
    'holidays',
    'a toy dinosaur',
    'Two astronauts',
    'A streetview image of a fastfood restaurant',
    'a cat',
    'a dog',
    'two cows',
]

# Checkpoint and tokenizer paths (absolute paths).
image_encoder_checkpoint = os.path.join(CKPT_DIR, CHECKPOINT_MAP[model][variant][0])
text_encoder_checkpoint = os.path.join(CKPT_DIR, CHECKPOINT_MAP[model][variant][1])
tokenizer_path = os.path.join(CKPT_DIR, 'tokenizer.model')

print(f'Image encoder checkpoint: {image_encoder_checkpoint}')
print(f'Text encoder checkpoint: {text_encoder_checkpoint}')
print(f'Tokenizer path: {tokenizer_path}')

# Verify files exist.
for p in [image_encoder_checkpoint, text_encoder_checkpoint, tokenizer_path]:
  assert os.path.exists(p), f'File not found: {p}. Please run the download cell first.'
print('\nAll files verified!')

### Data
Now that we have the model set up, let's load the training data. It consists of:

- images in `jpg` format:
```
https://dl.fbaipublicfiles.com/dinov3/notebooks/foreground_segmentation/foreground_segmentation_images.tar.gz
```

- and segmentation masks stored as alpha channels in `png` files:
```
https://dl.fbaipublicfiles.com/dinov3/notebooks/foreground_segmentation/foreground_segmentation_labels.tar.gz
```

In total, there are 9 training image / mask pairs.


In [ ]:
IMAGES_URI = "https://dl.fbaipublicfiles.com/dinov3/notebooks/foreground_segmentation/foreground_segmentation_images.tar.gz"
LABELS_URI = "https://dl.fbaipublicfiles.com/dinov3/notebooks/foreground_segmentation/foreground_segmentation_labels.tar.gz"

def load_images_from_remote_tar(tar_uri: str) -> list[Image.Image]:
    images = []
    with urllib.request.urlopen(tar_uri) as f:
        tar = tarfile.open(fileobj=io.BytesIO(f.read()))
        for member in tar.getmembers():
            image_data = tar.extractfile(member)
            image = Image.open(image_data)
            images.append(image)
    return images

images = load_images_from_remote_tar(IMAGES_URI)
labels = load_images_from_remote_tar(LABELS_URI)
n_images = len(images)
assert n_images == len(labels), f"{len(images)=}, {len(labels)=}"

print(f"Loaded {n_images} images and labels")

Let's, for example, visualize the first image / mask pair:

In [ ]:
data_index = 0

print(f"Showing image / mask at index {data_index}:")

image = images[data_index]
mask = labels[data_index]
foreground = Image.composite(image, mask, mask)
mask_bg_np = np.copy(np.array(mask))
mask_bg_np[:, :, 3] = 255 - mask_bg_np[:, :, 3]
mask_bg = Image.fromarray(mask_bg_np)
background = Image.composite(image, mask_bg, mask_bg)

data_to_show = [image, mask, foreground, background]
data_labels = ["Image", "Mask", "Foreground", "Background"]

plt.figure(figsize=(16, 4), dpi=300)
for i in range(len(data_to_show)):
    plt.subplot(1, len(data_to_show), i + 1)
    plt.imshow(data_to_show[i])
    plt.axis('off')
    plt.title(data_labels[i], fontsize=12)
plt.show()

### Building Per-Patch Label Map

Since our models run with a patch size of 16, we have to quantize the ground truth to a 16x16 pixels grid. To achieve this, we define:
- the resize transform to resize an image such that it aligns well with the 16x16 grid;
- a uniform 16x16 conv layer as a [box blur filter](https://en.wikipedia.org/wiki/Box_blur) with stride equal to the patch size.

In [ ]:
# quantization filter for the given patch size
patch_quant_filter = torch.nn.Conv2d(1, 1, PATCH_SIZE, stride=PATCH_SIZE, bias=False)
patch_quant_filter.weight.data.fill_(1.0 / (PATCH_SIZE * PATCH_SIZE))

# image resize transform to dimensions divisible by patch size
def resize_transform(
    mask_image: Image,
    image_size: int = image_size,
    patch_size: int = PATCH_SIZE,
) -> torch.Tensor:
    w, h = mask_image.size
    h_patches = int(image_size / patch_size)
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(TF.resize(mask_image, (h_patches * patch_size, w_patches * patch_size)))

Let's, for example, visualize the first mask before and after quantization:

In [ ]:
mask_0 = labels[0].split()[-1]
mask_0_resized = resize_transform(mask_0)
with torch.no_grad():
    mask_0_quantized = patch_quant_filter(mask_0_resized).squeeze().detach().cpu()

plt.figure(figsize=(4, 2), dpi=300)
plt.subplot(1, 2, 1)
plt.imshow(mask_0)
plt.axis('off')
plt.title(f"Original Mask, Size {mask_0.size}", fontsize=5)
plt.subplot(1, 2, 2)
plt.imshow(mask_0_quantized)
plt.axis('off')
plt.title(f"Quantized Mask, Size {tuple(mask_0_quantized.shape)}", fontsize=5)
plt.show()

### Extracting Features and Labels for All the Images
Now we will loop over the 9 training images, and extract for each image the patch labels, as well as the patch features.

In [ ]:
xs = []
ys = []
image_index = []

n_layers = MODEL_LAYERS_MAP[model][variant]
weights_image = dict(np.load(image_encoder_checkpoint, allow_pickle=False))
for key in weights_image:
  weights_image[key] = torch.tensor(weights_image[key])
ffn_layer = 'swiglu' if variant == 'g' else 'mlp'

embeddings_image, spatial_features = [], []

with torch.no_grad():
  # Load the vision encoder.
  vit_constructor_name = MODEL_CONSTRUCTOR_MAP[model][variant]
  vit_constructor = getattr(image_encoder, vit_constructor_name)
  model_image = vit_constructor(
      img_size=image_size,
      patch_size=PATCH_SIZE,
      ffn_layer=ffn_layer,
      block_chunks=0,
      init_values=1.0,
      interpolate_antialias=True,
      interpolate_offset=0.0,
  )
  model_image.load_state_dict(weights_image)
  model_image.to("cuda", dtype=torch.float32)

  for i in tqdm(range(n_images), desc="Processing images"):
      # Loading the ground truth
      mask_i = labels[i].split()[-1]
      mask_i_resized = resize_transform(mask_i)
      mask_i_quantized = patch_quant_filter(mask_i_resized).squeeze().view(-1).detach().cpu()
      ys.append(mask_i_quantized)
      # Loading the image data
      image_i = images[i].convert('RGB')
      image_i_resized = resize_transform(image_i)
      image_i_resized = TF.normalize(image_i_resized, mean=IMAGE_MEAN, std=IMAGE_STD)
      image_i_resized = image_i_resized.unsqueeze(0).cuda()

      feats = model_image.get_intermediate_layers(image_i_resized, n=range(n_layers), reshape=True, norm=True)
      dim = feats[-1].shape[1]
      xs.append(feats[-1].squeeze().view(dim, -1).permute(1,0).detach().cpu())

      image_index.append(i * torch.ones(ys[-1].shape))


# Concatenate all lists into torch tensors
xs = torch.cat(xs)
ys = torch.cat(ys)
image_index = torch.cat(image_index)

# keeping only the patches that have clear positive or negative label
idx = (ys < 0.01) | (ys > 0.99)
xs = xs[idx]
ys = ys[idx]
image_index = image_index[idx]

print("Design matrix of size : ", xs.shape)
print("Label matrix of size : ", ys.shape)

### Training a Classifier and Model Selection
We computed the features, let's now train a classifier! Our data is very strongly correlated image-by-image. Therefore, to do proper model selection, we can't simply split the data in an IID way. We need to do something a bit smarter. We will do leave-one-out, and consecutively exclude each image as a validation set.
We'll try 8 values of C ranging from 1e-7 to 1e-0.

For each value of C and each image, we plot the precision-recall curve of the classifier, and report the mAP (area under the PR curve).

In [ ]:
cs = np.logspace(-7, 0, 8)
scores = np.zeros((n_images, len(cs)))

for i in range(n_images):
    # We use leave-one-out so train will be all but image i, val will be image i
    print('validation using image_{:02d}.jpg'.format(i+1))
    train_selection = image_index != float(i)
    fold_x = xs[train_selection].numpy()
    fold_y = (ys[train_selection] > 0).long().numpy()
    val_x = xs[~train_selection].numpy()
    val_y = (ys[~train_selection] > 0).long().numpy()

    plt.figure()
    for j, c in enumerate(cs):
        print("training logisitic regression with C={:.2e}".format(c))
        clf = LogisticRegression(random_state=0, C=c, max_iter=10000).fit(fold_x, fold_y)
        output = clf.predict_proba(val_x)
        precision, recall, thresholds = precision_recall_curve(val_y, output[:, 1])
        s = average_precision_score(val_y, output[:, 1])
        scores[i, j] = s
        plt.plot(recall, precision, label='C={:.1e} AP={:.1f}'.format(c, s*100))

    plt.grid()
    plt.xlabel('recall')
    plt.title('image_{:02d}.jpg'.format(i+1))
    plt.ylabel('precision')
    plt.axis([0, 1, 0, 1])
    plt.legend()
    plt.show()


### Choosing the Best C
Now, let's have a look at which value of C works best on average. To this end we will plot the average mAP across all validation images.

In [ ]:
plt.figure(figsize=(3, 2), dpi=300)
plt.rcParams.update({
    "xtick.labelsize": 5,
    "ytick.labelsize": 5,
    "axes.labelsize": 5,
})
plt.plot(scores.mean(axis=0))
plt.xticks(np.arange(len(cs)), ["{:.0e}".format(c) for c in cs])
plt.xlabel('data fit C')
plt.ylabel('average AP')
plt.grid()
plt.show()

### Retraining with the optimal regularization
Given the above, we seem to have a winner: C=0.1.
Let's now train a model using this optimal data-fit value.

In [ ]:
clf = LogisticRegression(random_state=0, C=0.1, max_iter=100000, verbose=2).fit(xs.numpy(), (ys > 0).long().numpy())

### Test Images and Inference

We have a classifier, now it is time to test it! We will predict the probability of patch being foreground given an image, and then process it with a 3x3 median filter to smooth it out.

In [ ]:
test_image_fpath = "/content/images/example_image.jpg"

test_image = Image.open(test_image_fpath).convert("RGB")
test_image_resized = resize_transform(test_image)
test_image_normalized = TF.normalize(test_image_resized, mean=IMAGE_MEAN, std=IMAGE_STD)

with torch.inference_mode():
    with torch.autocast(device_type='cuda', dtype=torch.float32):
        feats = model_image.get_intermediate_layers(test_image_normalized.unsqueeze(0).cuda(), n=range(n_layers), reshape=True, norm=True)
        x = feats[-1].squeeze().detach().cpu()
        dim = x.shape[0]
        x = x.view(dim, -1).permute(1, 0)

h_patches, w_patches = [int(d / PATCH_SIZE) for d in test_image_resized.shape[1:]]

fg_score = clf.predict_proba(x)[:, 1].reshape(h_patches, w_patches)
fg_score_mf = torch.from_numpy(signal.medfilt2d(fg_score, kernel_size=3))

plt.figure(figsize=(9, 3), dpi=300)
plt.subplot(1, 3, 1)
plt.axis('off')
plt.imshow(test_image_resized.permute(1, 2, 0))
plt.title('input image')
plt.subplot(1, 3, 2)
plt.axis('off')
plt.imshow(fg_score)
plt.title('foreground score')
plt.subplot(1, 3, 3)
plt.axis('off')
plt.imshow(fg_score_mf)
plt.title('+ median filter')
plt.show()

In [ ]:
save_root = '.'
model_path = os.path.join(save_root, "fg_classifier.pkl")
with open(model_path, "wb") as f:
  pickle.dump(clf, f)